In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  03_analise.ipynb — Análises e visualizações (Plotly)      ║
# ║  TCC: Cobertura Vacinal no Piauí (2015–2022)               ║
# ║  Dani Alcoforado — Tecnólogo em Ciência de Dados (UNINTER) ║
# ╚══════════════════════════════════════════════════════════════╝
#
# ANÁLISES:
#   A1 — Evolução temporal do Piauí (doses infantis 2015–2022)
#   A2 — Impacto COVID: queda e recuperação por macrorregião
#   A3 — Ranking de municípios críticos (abaixo da meta 95%)
#   A4 — Parnaíba em detalhe vs média estadual
#
# PRÉ-REQUISITO: 02_limpeza.ipynb concluído
#   dados_tratados/pni_piaui_unificado.parquet
#   dados_tratados/pni_piaui_clean.parquet

In [4]:
# ── CÉLULA 1: Setup ───────────────────────────────────────────
!pip install plotly kaleido --quiet

from google.colab import drive
drive.mount('/content/drive')

import os, warnings
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

RAIZ           = '/content/drive/MyDrive/TCC_Vacinal_Piaui'
DADOS_TRATADOS = f'{RAIZ}/dados_tratados'
DADOS_BRUTOS   = f'{RAIZ}/dados_brutos'
FIGURAS        = f'{RAIZ}/figuras'
os.makedirs(FIGURAS, exist_ok=True)

# ── Paleta e tema do projeto ──────────────────────────────────
COR_PIAUI    = '#2196F3'   # azul — estado
COR_PARNAIBA = '#FF6B35'   # laranja — Parnaíba
COR_META     = '#4CAF50'   # verde — meta 95%
COR_ALERTA   = '#F44336'   # vermelho — crítico
COR_COVID    = '#9C27B0'   # roxo — período COVID
COR_FUNDO    = '#F8F9FA'

TEMPLATE = dict(
    layout=dict(
        font=dict(family='Inter, Arial, sans-serif', size=13),
        plot_bgcolor='white',
        paper_bgcolor='white',
        title_font_size=16,
        title_font_color='#1a1a2e',
        margin=dict(t=80, b=60, l=70, r=40),
        legend=dict(bgcolor='rgba(255,255,255,0.8)', bordercolor='#ddd', borderwidth=1),
    )
)

def salvar_fig(fig, nome):
    """Salva figura como HTML interativo e PNG estático."""
    fig.write_html(f'{FIGURAS}/{nome}.html')
    try:
        fig.write_image(f'{FIGURAS}/{nome}.png', width=1200, height=600, scale=2)
        print(f'  💾 {nome}.html + {nome}.png')
    except Exception:
        print(f'  💾 {nome}.html (PNG requer kaleido — instale se necessário)')

print('✅ Setup completo')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup completo


In [7]:
# ── CÉLULA 2: Carregar dados ──────────────────────────────────

# Dataset agregado: 1 linha por município × ano
df = pd.read_parquet(f'{DADOS_TRATADOS}/pni_piaui_unificado.parquet')

if 'dado_incompleto' not in df.columns:
    df['dado_incompleto'] = df['ano'] == 2019

# Dataset detalhado PySUS: vacina × faixa × cobertura% (2015–2019)
df_det = pd.read_parquet(f'{DADOS_TRATADOS}/pni_piaui_clean.parquet')

# Parnaíba
COD_PARNAIBA = '220770'
df_par = df[df['cod_municipio'] == COD_PARNAIBA].copy()
df_pi  = df[df['cod_municipio'] != COD_PARNAIBA].copy()

# Excluir 2019 incompleto das análises de tendência
df_ok  = df[df['dado_incompleto'] == False].copy()
ANOS_ANALISE = sorted(df_ok['ano'].unique())

print(f'Dataset agregado: {len(df):,} linhas | {df["ano"].nunique()} anos | {df["cod_municipio"].nunique()} municípios')
print(f'Dataset detalhe:  {len(df_det):,} linhas')
print(f'Anos de análise (sem 2019 incompleto): {ANOS_ANALISE}')
print(f'\nParnaíba: {len(df_par)} registros')

Dataset agregado: 1,792 linhas | 8 anos | 224 municípios
Dataset detalhe:  855,767 linhas
Anos de análise (sem 2019 incompleto): [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2020), np.int64(2021), np.int64(2022)]

Parnaíba: 8 registros


In [8]:
# ══════════════════════════════════════════════════════════════
# A1 — EVOLUÇÃO TEMPORAL DO PIAUÍ (2015–2022)
# ══════════════════════════════════════════════════════════════

# Agregar por ano — estado inteiro
ev_pi = (
    df_ok
    .groupby(['ano', 'fonte'], as_index=False)
    .agg(
        doses_infantil = ('doses_infantil', 'sum'),
        doses_total    = ('doses_total',    'sum'),
        cobertura_media = ('cobertura_media', 'mean'),
    )
)

# Calcular variação vs 2018
base_2018 = ev_pi.loc[ev_pi['ano'] == 2018, 'doses_infantil'].values[0]
ev_pi['var_vs_2018'] = ((ev_pi['doses_infantil'] / base_2018) - 1) * 100

# ── Gráfico A1: linha dupla doses + cobertura ─────────────────
fig_a1 = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=[
        'Doses infantis aplicadas (< 5 anos) — Piauí',
        'Cobertura vacinal média (%) — PySUS 2015–2018'
    ]
)

# Separar pré-COVID e COVID
pre  = ev_pi[ev_pi['fonte'] == 'PySUS']
pos  = ev_pi[ev_pi['fonte'] == 'TabNet_doses']

# Linha de doses pré-COVID
fig_a1.add_trace(go.Scatter(
    x=pre['ano'], y=pre['doses_infantil'],
    mode='lines+markers',
    name='Doses (PySUS)',
    line=dict(color=COR_PIAUI, width=3),
    marker=dict(size=8),
    hovertemplate='%{x}: %{y:,.0f} doses<extra></extra>',
), row=1, col=1)

# Linha de doses pós-COVID
fig_a1.add_trace(go.Scatter(
    x=list(pre['ano'])[-1:] + list(pos['ano']),
    y=list(pre['doses_infantil'])[-1:] + list(pos['doses_infantil']),
    mode='lines+markers',
    name='Doses (TabNet)',
    line=dict(color=COR_COVID, width=3, dash='dash'),
    marker=dict(size=8, symbol='diamond'),
    hovertemplate='%{x}: %{y:,.0f} doses<extra></extra>',
), row=1, col=1)

# Banda COVID
fig_a1.add_vrect(
    x0=2019.5, x1=2022.5,
    fillcolor=COR_COVID, opacity=0.07,
    layer='below', line_width=0,
    annotation_text='Período COVID-19',
    annotation_position='top left',
    annotation_font_color=COR_COVID,
    row=1, col=1
)

# Linha baseline 2018
fig_a1.add_hline(
    y=base_2018, line_dash='dot',
    line_color='gray', opacity=0.5,
    annotation_text=f'Baseline 2018: {base_2018:,.0f}',
    annotation_position='right',
    row=1, col=1
)

# Cobertura (só PySUS)
fig_a1.add_trace(go.Scatter(
    x=pre['ano'], y=pre['cobertura_media'],
    mode='lines+markers',
    name='Cobertura média %',
    line=dict(color=COR_PIAUI, width=3),
    marker=dict(size=8),
    hovertemplate='%{x}: %{y:.1f}%<extra></extra>',
), row=2, col=1)

# Linha meta 95%
fig_a1.add_hline(
    y=95, line_dash='dash',
    line_color=COR_META, line_width=2,
    annotation_text='Meta PNI: 95%',
    annotation_font_color=COR_META,
    row=2, col=1
)

fig_a1.update_layout(
    **TEMPLATE['layout'],
    title='A1 — Evolução da vacinação infantil no Piauí (2015–2022)',
    height=600,
)
fig_a1.update_yaxes(title_text='Doses aplicadas', row=1, col=1)
fig_a1.update_yaxes(title_text='Cobertura (%)', row=2, col=1, range=[0, 120])
fig_a1.update_xaxes(title_text='Ano', dtick=1, row=2, col=1)

fig_a1.show()
salvar_fig(fig_a1, 'A1_evolucao_temporal_piaui')

# Tabela resumo
print('\nResumo A1:')
print(ev_pi[['ano','fonte','doses_infantil','var_vs_2018','cobertura_media']]
      .rename(columns={'doses_infantil':'Doses infantil',
                        'var_vs_2018':'Var% vs 2018',
                        'cobertura_media':'Cob. média %'})
      .to_string(index=False, float_format=lambda x: f'{x:.1f}'))

  💾 A1_evolucao_temporal_piaui.html (PNG requer kaleido — instale se necessário)

Resumo A1:
 ano        fonte  Doses infantil  Var% vs 2018  Cob. média %
2015        PySUS       1461516.0          38.2          61.6
2016        PySUS       1661172.0          57.0          65.6
2017        PySUS        939531.0         -11.2          69.2
2018        PySUS       1057811.0           0.0          74.1
2020 TabNet_doses       1020289.0          -3.5           NaN
2021 TabNet_doses        957673.0          -9.5           NaN
2022 TabNet_doses       1124719.0           6.3           NaN


In [9]:
# ══════════════════════════════════════════════════════════════
# A2 — IMPACTO COVID: QUEDA E RECUPERAÇÃO POR MUNICÍPIO
# ══════════════════════════════════════════════════════════════

# Calcular variação de doses infantis em relação a 2018 (baseline)
base_mun = (
    df_ok[df_ok['ano'] == 2018]
    [['cod_municipio', 'nome_municipio', 'doses_infantil']]
    .rename(columns={'doses_infantil': 'doses_base_2018'})
)

df_covid = (
    df_ok[df_ok['ano'].isin([2020, 2021, 2022])]
    .merge(base_mun, on='cod_municipio', how='left')
)
df_covid['var_pct'] = (
    (df_covid['doses_infantil'] / df_covid['doses_base_2018'] - 1) * 100
).round(1)

# Classificar impacto
def classif_impacto(v):
    if pd.isna(v):   return 'Sem dado'
    if v < -30:      return 'Queda grave (> 30%)'
    if v < -10:      return 'Queda moderada (10–30%)'
    if v < 0:        return 'Queda leve (< 10%)'
    if v < 10:       return 'Estável'
    return 'Recuperação (> 10%)'

df_covid['impacto'] = df_covid['var_pct'].apply(classif_impacto)

ordem_impacto = ['Queda grave (> 30%)', 'Queda moderada (10–30%)',
                  'Queda leve (< 10%)', 'Estável', 'Recuperação (> 10%)', 'Sem dado']
cores_impacto = {
    'Queda grave (> 30%)':     '#B71C1C',
    'Queda moderada (10–30%)': '#F44336',
    'Queda leve (< 10%)':      '#FF8A65',
    'Estável':                 '#90A4AE',
    'Recuperação (> 10%)':     '#43A047',
    'Sem dado':                '#E0E0E0',
}

# ── Gráfico A2a: distribuição do impacto por ano ──────────────
contagem = (
    df_covid.groupby(['ano', 'impacto'])
    .size().reset_index(name='n_municipios')
)
contagem['impacto'] = pd.Categorical(contagem['impacto'], categories=ordem_impacto, ordered=True)
contagem = contagem.sort_values(['ano', 'impacto'])

fig_a2a = px.bar(
    contagem, x='ano', y='n_municipios',
    color='impacto',
    color_discrete_map=cores_impacto,
    category_orders={'impacto': ordem_impacto},
    barmode='stack',
    title='A2 — Distribuição do impacto COVID nas doses infantis por município',
    labels={'n_municipios': 'Nº de municípios', 'ano': 'Ano', 'impacto': 'Impacto vs 2018'},
    text='n_municipios',
)
fig_a2a.update_layout(**TEMPLATE['layout'], height=480)
fig_a2a.update_traces(textposition='inside', textfont_size=11)
fig_a2a.update_xaxes(tickvals=[2020, 2021, 2022])
fig_a2a.show()
salvar_fig(fig_a2a, 'A2a_impacto_covid_distribuicao')

# ── Gráfico A2b: ranking dos 20 municípios com maior queda em 2021 ──
pior_2021 = (
    df_covid[df_covid['ano'] == 2021]
    .nsmallest(20, 'var_pct')
    [['nome_municipio_x', 'var_pct', 'doses_infantil', 'doses_base_2018']]
    .rename(columns={'nome_municipio_x': 'Município'})
    .sort_values('var_pct')
)

fig_a2b = px.bar(
    pior_2021, x='var_pct', y='Município',
    orientation='h',
    color='var_pct',
    color_continuous_scale=['#B71C1C', '#FF8A65', '#90A4AE'],
    title='A2 — 20 municípios com maior queda de doses infantis em 2021 vs 2018',
    labels={'var_pct': 'Variação % vs 2018', 'Município': ''},
)
fig_a2b.add_vline(x=0, line_color='black', line_width=1)
fig_a2b.update_layout(**TEMPLATE['layout'], height=580, coloraxis_showscale=False)
fig_a2b.update_traces(
    hovertemplate='%{y}<br>Variação: %{x:.1f}%<extra></extra>'
)
fig_a2b.show()
salvar_fig(fig_a2b, 'A2b_municipios_maior_queda_2021')

# Estatísticas
print('\nEstatísticas A2:')
for ano in [2020, 2021, 2022]:
    sub = df_covid[df_covid['ano'] == ano]['var_pct'].dropna()
    n_queda = (sub < 0).sum()
    n_grave = (sub < -30).sum()
    print(f'  {ano}: mediana={sub.median():+.1f}% | {n_queda}/224 municípios com queda | {n_grave} com queda grave')

  💾 A2a_impacto_covid_distribuicao.html (PNG requer kaleido — instale se necessário)


  💾 A2b_municipios_maior_queda_2021.html (PNG requer kaleido — instale se necessário)

Estatísticas A2:
  2020: mediana=-1.6% | 118/224 municípios com queda | 24 com queda grave
  2021: mediana=-12.6% | 168/224 municípios com queda | 41 com queda grave
  2022: mediana=+2.4% | 100/224 municípios com queda | 9 com queda grave


In [10]:
# ══════════════════════════════════════════════════════════════
# A3 — RANKING MUNICÍPIOS CRÍTICOS (ABAIXO DA META 95%)
# Usa dados PySUS 2015–2018 onde cobertura % está disponível
# ══════════════════════════════════════════════════════════════

META = 95.0

# Cobertura média por município no período PySUS
cob_mun = (
    df_det[df_det['cobertura_pct'].notna()]
    .groupby(['cod_municipio', 'nome_municipio', 'ano'], as_index=False)
    ['cobertura_pct'].mean()
    .rename(columns={'cobertura_pct': 'cobertura_media'})
)

# Anos abaixo da meta por município (frequência de anos críticos)
criticos = (
    cob_mun[cob_mun['cobertura_media'] < META]
    .groupby(['cod_municipio', 'nome_municipio'], as_index=False)
    .agg(
        anos_abaixo_meta = ('ano', 'count'),
        cobertura_minima = ('cobertura_media', 'min'),
        cobertura_media  = ('cobertura_media', 'mean'),
    )
    .sort_values(['anos_abaixo_meta', 'cobertura_media'])
)

# Top 25 mais críticos (maior frequência + menor cobertura)
top25 = criticos.nlargest(25, 'anos_abaixo_meta').sort_values('cobertura_media')

# Marcar Parnaíba
top25['cor'] = top25['cod_municipio'].apply(
    lambda x: COR_PARNAIBA if x == COD_PARNAIBA else COR_ALERTA
)
top25['label'] = top25['nome_municipio'].apply(
    lambda x: f'<b>{x}</b>' if x == 'Parnaíba' else x
)

# ── Gráfico A3a: ranking de cobertura média ───────────────────
fig_a3a = go.Figure()

fig_a3a.add_trace(go.Bar(
    x=top25['cobertura_media'],
    y=top25['nome_municipio'],
    orientation='h',
    marker_color=top25['cor'],
    text=top25['cobertura_media'].apply(lambda x: f'{x:.1f}%'),
    textposition='outside',
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Cobertura média: %{x:.1f}%<br>'
        '<extra></extra>'
    ),
))

fig_a3a.add_vline(
    x=META, line_dash='dash',
    line_color=COR_META, line_width=2,
    annotation_text=f'Meta: {META}%',
    annotation_font_color=COR_META,
)

fig_a3a.update_layout(
    **TEMPLATE['layout'],
    title='A3 — Top 25 municípios com maior frequência de anos abaixo da meta (2015–2018)',
    xaxis_title='Cobertura vacinal média (%)',
    yaxis_title='',
    height=650,
    xaxis_range=[0, 115],
    showlegend=False,
)
fig_a3a.show()
salvar_fig(fig_a3a, 'A3a_ranking_municipios_criticos')

# ── Gráfico A3b: heatmap cobertura × ano (top 20 críticos) ───
top20_cods = criticos.nlargest(20, 'anos_abaixo_meta')['cod_municipio'].tolist()

heat = (
    cob_mun[cob_mun['cod_municipio'].isin(top20_cods)]
    .pivot(index='nome_municipio', columns='ano', values='cobertura_media')
    .fillna(0)
)

fig_a3b = go.Figure(go.Heatmap(
    z=heat.values,
    x=heat.columns.tolist(),
    y=heat.index.tolist(),
    colorscale=[
        [0.0,  '#B71C1C'],
        [0.5,  '#FF8A65'],
        [0.95, '#FFF9C4'],
        [1.0,  '#1B5E20'],
    ],
    zmid=META,
    zmin=0, zmax=150,
    text=heat.values.round(1),
    texttemplate='%{text}%',
    textfont_size=10,
    hovertemplate='<b>%{y}</b><br>%{x}: %{z:.1f}%<extra></extra>',
    colorbar=dict(title='Cobertura %', ticksuffix='%'),
))

fig_a3b.update_layout(
    **TEMPLATE['layout'],
    title='A3 — Cobertura vacinal por ano (top 20 municípios críticos)',
    xaxis_title='Ano',
    yaxis_title='',
    height=560,
)
fig_a3b.show()
salvar_fig(fig_a3b, 'A3b_heatmap_municipios_criticos')

print(f'\nEstatísticas A3:')
print(f'  Municípios com pelo menos 1 ano abaixo da meta: {len(criticos)}')
print(f'  Municípios abaixo da meta em TODOS os anos (2015–2018): {(criticos["anos_abaixo_meta"] >= 4).sum()}')
print(f'  Cobertura média do estado: {cob_mun["cobertura_media"].mean():.1f}%')

parnaiba_crit = criticos[criticos['cod_municipio'] == COD_PARNAIBA]
if len(parnaiba_crit) > 0:
    p = parnaiba_crit.iloc[0]
    print(f'\n  Parnaíba: {p["anos_abaixo_meta"]} anos abaixo da meta | cobertura média: {p["cobertura_media"]:.1f}%')
else:
    print('\n  Parnaíba: acima da meta em todos os anos ✅')

  💾 A3a_ranking_municipios_criticos.html (PNG requer kaleido — instale se necessário)


  💾 A3b_heatmap_municipios_criticos.html (PNG requer kaleido — instale se necessário)

Estatísticas A3:
  Municípios com pelo menos 1 ano abaixo da meta: 224
  Municípios abaixo da meta em TODOS os anos (2015–2018): 207
  Cobertura média do estado: 58.2%

  Parnaíba: 5 anos abaixo da meta | cobertura média: 58.3%


In [11]:
# ══════════════════════════════════════════════════════════════
# A4 — PARNAÍBA EM DETALHE VS MÉDIA ESTADUAL
# ══════════════════════════════════════════════════════════════

# Série temporal completa
ev_par = df_par[df_par['dado_incompleto'] == False].copy()
ev_pi_agg = df_ok.groupby('ano', as_index=False).agg(
    doses_infantil_pi = ('doses_infantil', 'sum'),
    cobertura_pi      = ('cobertura_media', 'mean'),
)

# Normalizar por municípios (média por município)
ev_pi_agg['doses_media_mun'] = ev_pi_agg['doses_infantil_pi'] / 224

# ── Gráfico A4a: doses infantis Parnaíba vs média estadual ───
fig_a4a = go.Figure()

fig_a4a.add_trace(go.Scatter(
    x=ev_par['ano'], y=ev_par['doses_infantil'],
    mode='lines+markers',
    name='Parnaíba',
    line=dict(color=COR_PARNAIBA, width=3),
    marker=dict(size=9),
    hovertemplate='Parnaíba %{x}: %{y:,.0f} doses<extra></extra>',
))

fig_a4a.add_trace(go.Scatter(
    x=ev_pi_agg['ano'], y=ev_pi_agg['doses_media_mun'],
    mode='lines+markers',
    name='Média estadual (por município)',
    line=dict(color=COR_PIAUI, width=2, dash='dot'),
    marker=dict(size=7),
    hovertemplate='PI médio %{x}: %{y:,.0f} doses<extra></extra>',
))

fig_a4a.add_vrect(
    x0=2019.5, x1=2022.5,
    fillcolor=COR_COVID, opacity=0.07,
    layer='below', line_width=0,
    annotation_text='COVID-19',
    annotation_font_color=COR_COVID,
)

fig_a4a.update_layout(
    **TEMPLATE['layout'],
    title='A4 — Doses infantis: Parnaíba vs média estadual por município (2015–2022)',
    xaxis_title='Ano',
    yaxis_title='Doses aplicadas (< 5 anos)',
    xaxis=dict(dtick=1),
    height=460,
)
fig_a4a.show()
salvar_fig(fig_a4a, 'A4a_parnaiba_vs_estadual_doses')

# ── Gráfico A4b: cobertura % Parnaíba vs estado (2015–2018) ──
cob_par_det = (
    df_det[
        (df_det['cod_municipio'] == COD_PARNAIBA) &
        (df_det['cobertura_pct'].notna())
    ]
    .groupby('ano', as_index=False)['cobertura_pct'].mean()
    .rename(columns={'cobertura_pct': 'cob_parnaiba'})
)

cob_pi_det = (
    df_det[df_det['cobertura_pct'].notna()]
    .groupby('ano', as_index=False)['cobertura_pct'].mean()
    .rename(columns={'cobertura_pct': 'cob_piaui'})
)

cob_comp = cob_par_det.merge(cob_pi_det, on='ano')

fig_a4b = go.Figure()

fig_a4b.add_trace(go.Bar(
    x=cob_comp['ano'], y=cob_comp['cob_parnaiba'],
    name='Parnaíba', marker_color=COR_PARNAIBA, opacity=0.85,
    hovertemplate='Parnaíba %{x}: %{y:.1f}%<extra></extra>',
))
fig_a4b.add_trace(go.Bar(
    x=cob_comp['ano'], y=cob_comp['cob_piaui'],
    name='Piauí (média)', marker_color=COR_PIAUI, opacity=0.85,
    hovertemplate='Piauí %{x}: %{y:.1f}%<extra></extra>',
))

fig_a4b.add_hline(
    y=META, line_dash='dash',
    line_color=COR_META, line_width=2,
    annotation_text=f'Meta: {META}%',
    annotation_font_color=COR_META,
)

fig_a4b.update_layout(
    **TEMPLATE['layout'],
    title='A4 — Cobertura vacinal média: Parnaíba vs Piauí (2015–2018)',
    xaxis_title='Ano',
    yaxis_title='Cobertura vacinal média (%)',
    barmode='group',
    xaxis=dict(tickvals=cob_comp['ano'].tolist(), dtick=1),
    height=420,
)
fig_a4b.show()
salvar_fig(fig_a4b, 'A4b_parnaiba_vs_estadual_cobertura')

# ── Gráfico A4c: evolução de doses por vacina em Parnaíba ─────
top_vacinas = (
    df_det[
        (df_det['cod_municipio'] == COD_PARNAIBA) &
        (df_det['doses_aplicadas'].notna()) &
        (df_det['doses_aplicadas'] > 0)
    ]
    .groupby('vacina_nome')['doses_aplicadas'].sum()
    .nlargest(8).index.tolist()
)

vac_par = (
    df_det[
        (df_det['cod_municipio'] == COD_PARNAIBA) &
        (df_det['vacina_nome'].isin(top_vacinas)) &
        (df_det['doses_aplicadas'].notna())
    ]
    .groupby(['ano', 'vacina_nome'], as_index=False)['doses_aplicadas'].sum()
)

fig_a4c = px.line(
    vac_par, x='ano', y='doses_aplicadas',
    color='vacina_nome',
    markers=True,
    title='A4 — Doses por vacina em Parnaíba (top 8, 2015–2019)',
    labels={
        'doses_aplicadas': 'Doses aplicadas',
        'ano': 'Ano',
        'vacina_nome': 'Vacina'
    },
)
fig_a4c.update_layout(**TEMPLATE['layout'], height=460, xaxis=dict(dtick=1))
fig_a4c.show()
salvar_fig(fig_a4c, 'A4c_parnaiba_doses_por_vacina')

# Estatísticas
print('\nEstatísticas A4 — Parnaíba:')
par_2018 = ev_par[ev_par['ano'] == 2018]
par_2020 = ev_par[ev_par['ano'] == 2020]
par_2022 = ev_par[ev_par['ano'] == 2022]

if len(par_2018) and len(par_2020):
    queda = (par_2020['doses_infantil'].values[0] / par_2018['doses_infantil'].values[0] - 1) * 100
    print(f'  Queda 2020 vs 2018: {queda:+.1f}%')
if len(par_2018) and len(par_2022):
    recup = (par_2022['doses_infantil'].values[0] / par_2018['doses_infantil'].values[0] - 1) * 100
    print(f'  Variação 2022 vs 2018: {recup:+.1f}%')

if len(cob_comp):
    print(f'  Cobertura Parnaíba vs Piauí:')
    for _, row in cob_comp.iterrows():
        diff = row['cob_parnaiba'] - row['cob_piaui']
        print(f'    {int(row["ano"])}: Parnaíba {row["cob_parnaiba"]:.1f}% | PI {row["cob_piaui"]:.1f}% | diff {diff:+.1f}pp')

  💾 A4a_parnaiba_vs_estadual_doses.html (PNG requer kaleido — instale se necessário)


  💾 A4b_parnaiba_vs_estadual_cobertura.html (PNG requer kaleido — instale se necessário)


  💾 A4c_parnaiba_doses_por_vacina.html (PNG requer kaleido — instale se necessário)

Estatísticas A4 — Parnaíba:
  Queda 2020 vs 2018: -47.2%
  Variação 2022 vs 2018: +3.2%
  Cobertura Parnaíba vs Piauí:
    2015: Parnaíba 75.0% | PI 61.9% | diff +13.1pp
    2016: Parnaíba 64.6% | PI 65.8% | diff -1.2pp
    2017: Parnaíba 73.7% | PI 69.2% | diff +4.5pp
    2018: Parnaíba 68.3% | PI 74.0% | diff -5.7pp
    2019: Parnaíba 9.9% | PI 20.4% | diff -10.5pp


In [12]:
# ── CÉLULA FINAL: Sumário de arquivos gerados ─────────────────

print('╔══════════════════════════════════════════════════════╗')
print('║  ✅  03_analise.ipynb — CONCLUÍDO                   ║')
print('╠══════════════════════════════════════════════════════╣')
print('║  Figuras geradas:                                   ║')
figs = [
    'A1_evolucao_temporal_piaui',
    'A2a_impacto_covid_distribuicao',
    'A2b_municipios_maior_queda_2021',
    'A3a_ranking_municipios_criticos',
    'A3b_heatmap_municipios_criticos',
    'A4a_parnaiba_vs_estadual_doses',
    'A4b_parnaiba_vs_estadual_cobertura',
    'A4c_parnaiba_doses_por_vacina',
]
for f in figs:
    html_ok = os.path.exists(f'{FIGURAS}/{f}.html')
    png_ok  = os.path.exists(f'{FIGURAS}/{f}.png')
    status  = '✅' if html_ok else '❌'
    tipos   = 'HTML+PNG' if (html_ok and png_ok) else ('HTML' if html_ok else 'N/A')
    print(f'║  {status} {f[:42]:<42} {tipos:>7} ║')
print('╠══════════════════════════════════════════════════════╣')
print('║  PRÓXIMO: 04_streamlit.ipynb                        ║')
print('╚══════════════════════════════════════════════════════╝')

╔══════════════════════════════════════════════════════╗
║  ✅  03_analise.ipynb — CONCLUÍDO                   ║
╠══════════════════════════════════════════════════════╣
║  Figuras geradas:                                   ║
║  ✅ A1_evolucao_temporal_piaui                    HTML ║
║  ✅ A2a_impacto_covid_distribuicao                HTML ║
║  ✅ A2b_municipios_maior_queda_2021               HTML ║
║  ✅ A3a_ranking_municipios_criticos               HTML ║
║  ✅ A3b_heatmap_municipios_criticos               HTML ║
║  ✅ A4a_parnaiba_vs_estadual_doses                HTML ║
║  ✅ A4b_parnaiba_vs_estadual_cobertura            HTML ║
║  ✅ A4c_parnaiba_doses_por_vacina                 HTML ║
╠══════════════════════════════════════════════════════╣
║  PRÓXIMO: 04_streamlit.ipynb                        ║
╚══════════════════════════════════════════════════════╝
